
The dataset is here:  
https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023

And the folder with all the product datasets is here:  
https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/tree/main/raw/meta_categories

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import numpy as np
import random
from pricer.items import Item
from pricer.parser import parse
load_dotenv(override=True)

In [ ]:

hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Appliances", split="full", trust_remote_code=True)
print(f"Number of Appliances: {len(dataset):,}")
dataset[6]

In [ ]:
max_price = 0
max_item = None

for datapoint in tqdm(dataset):    #tqdm is a amll pythonm library that adds progress bar.
    try:
        price = float(datapoint["price"])
        if price > max_price:
            max_item = datapoint
            max_price = price
    except ValueError:
        pass

print(f"The most expensive item is {max_item['title']} and it costs {max_price:,.2f}")


https://www.amazon.com/TurboChef-Electric-Countertop-Microwave-Convection/dp/B01D05U9NO/

In [ ]:
# Load into Item objects if they have a price range $1-$1000 and enough details

items = [parse(datapoint, "Appliances") for datapoint in tqdm(dataset)]
items = [item for item in items if item is not None]
print(f"There are {len(items):,} items from {len(dataset):,} datapoints")

In [ ]:
items[0]

In [ ]:
prices = [item.price for item in items]
lengths = [len(item.full) for item in items]

In [ ]:
# Plot the distribution of lengths

plt.figure(figsize=(15, 6))
plt.title(f"Lengths: Avg {sum(lengths)/len(lengths):,.0f} and highest {max(lengths):,}\n")
plt.xlabel('Length (chars)')
plt.ylabel('Count')
plt.hist(lengths, rwidth=0.7, color="lightblue", bins=range(0, 6000, 100))
plt.show()

In [ ]:
max_length = max(lengths)
max_length_item = items[lengths.index(max_length)]
print(max_length_item.full)


In [ ]:
# Plot the distribution of prices
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.2f} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="orange", bins=range(0, 1000, 10))
plt.show()

In [ ]:
print(items[3].full)

In [ ]:
from pricer.loaders import ItemLoader
loader = ItemLoader("Appliances")
items = loader.load()


In [ ]:
dataset_names = [
    "Automotive",
    "Electronics",
    "Office_Products",
    "Tools_and_Home_Improvement",
    "Cell_Phones_and_Accessories",
    "Toys_and_Games",
    "Appliances",
    "Musical_Instruments",
]

In [ ]:
items = []
for dataset_name in dataset_names:
    loader = ItemLoader(dataset_name)
    items.extend(loader.load())

In [ ]:
print(f"A grand total of {len(items):,} items")

In [ ]:
items[1000]

In [ ]:
random.seed(42)
random.shuffle(items)

seen = set()
items = [x for x in tqdm(items) if not (x.title in seen or seen.add(x.title))]

seen = set()
items = [x for x in tqdm(items) if not (x.full in seen or seen.add(x.full))]

del seen
print(f"After deduplication, we have {len(items):,} items")

In [ ]:
lengths = [len(item.full) for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Text length: Avg {sum(lengths)/len(lengths):,.1f} and highest {max(lengths):,}\n")
plt.xlabel('Length (characters)')
plt.ylabel('Count')
plt.hist(lengths, rwidth=0.7, color="skyblue", bins=range(0, 4050, 50))
plt.show()

In [ ]:
# Plot the distribution of prices

prices = [item.price for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
from collections import Counter
category_counts = Counter([item.category for item in items])

categories = category_counts.keys()
counts = [category_counts[category] for category in categories]

plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title('How many in each category')
plt.xlabel('Categories')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')

for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')

plt.show()

In [ ]:
np.random.seed(42)

SIZE = 820_000

prices = np.array([it.price for it in items], dtype=float)
categories = np.array([it.category for it in items])
p = (prices - prices.min()) / (prices.max() - prices.min() + 1e-9)

w = p**2
w[categories == "Tools_and_Home_Improvement"] *= 0.5
w[categories == "Automotive"] *= 0.05

w = w / w.sum()
idx = np.random.choice(len(items), size=SIZE, replace=False, p=w)
sample = [items[i] for i in idx]

In [ ]:
prices = [item.price for item in sample]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} lowest {min(prices):,} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
# Just for good measure, let's shuffle the sample again for the final dataset

random.seed(42)
random.shuffle(sample)

In [ ]:
prices = [item.price for item in sample]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} lowest {min(prices):,} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
from collections import Counter
category_counts = Counter([item.category for item in sample])

categories = category_counts.keys()
counts = [category_counts[category] for category in categories]

# Bar chart by category
plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title('How many in each category')
plt.xlabel('Categories')
plt.ylabel('Count')

plt.xticks(rotation=30, ha='right')

# Add value labels on top of each bar
for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')

# Display the chart
plt.show()

In [ ]:
# Automotive still in the lead, but improved somewhat
# For another perspective, let's look at a pie

plt.figure(figsize=(12, 10))
plt.pie(counts, labels=categories, autopct='%1.0f%%', startangle=90)

# Add a circle at the center to create a donut chart (optional)
centre_circle = plt.Circle((0,0), 0.70, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)
plt.title('Categories')

# Equal aspect ratio ensures that pie is drawn as a circle
plt.axis('equal')  

plt.show()

In [ ]:
# How does the price vary with the character count?

sizes = [len(item.full) for item in sample]
prices = [item.price for item in sample]

# Create the scatter plot
plt.figure(figsize=(15, 8))
plt.scatter(sizes, prices, s=0.2, color="red")

# Add labels and title
plt.xlabel('Size')
plt.ylabel('Price')
plt.title('Is there a simple correlation with text length?')

# Display the plot
plt.show()

In [ ]:
# How does the price vary with the weight?

ounces = [item.weight for item in sample]
prices = [item.price for item in sample]

# Create the scatter plot
plt.figure(figsize=(15, 8))
plt.scatter(ounces, prices, s=0.2, color="darkorange")

# Add labels and title
plt.xlabel('Weight (ounces)')
plt.ylabel('Price')
plt.xlim(0, 400)
plt.title('Is there a simple correlation with weight?')

# Display the plot
plt.show()

In [ ]:
username = "ed-donner"
full = f"{username}/items_raw_full"
lite = f"{username}/items_raw_lite"

train = sample[:800_000]
val = sample[800_000:810_000]
test = sample[810_000:]

Item.push_to_hub(full, train, val, test)

train_lite = train[:20_000]
val_lite = val[:1_000]
test_lite = test[:1_000]

Item.push_to_hub(lite, train_lite, val_lite, test_lite)

Here is a detailed, cell-by-cell breakdown of everything that happens in Data_curation.ipynb:

1. Markdown Cell Provides links to the Hugging Face repository for the raw Amazon-Reviews-2023 dataset and its category folders.

2. Code Cell Imports all the necessary Python libraries for the notebook, including:

Standard libraries: os, random
Data/ML libraries: numpy, matplotlib.pyplot, datasets (Hugging Face)
Utility libraries: dotenv, huggingface_hub, tqdm
Custom modules: Item and parse from a local pricer package. It also loads environment variables using load_dotenv().
3. Code Cell Retrieves the Hugging Face access token from the environment variables (HF_TOKEN) and authenticates with Hugging Face so the private/gated datasets can be downloaded.

4. Code Cell Loads the "Appliances" split of the raw Amazon metadata dataset using Hugging Face's load_dataset. It then prints out the total number of appliance records and displays the 7th item in the dataset (dataset[6]).

5. Code Cell Iterates through the entire appliances dataset to find the single most expensive item. It attempts to parse the "price" field as a float (ignoring any records where the price is missing or invalid) and keeps track of the maximum value encountered. Finally, it prints the title and price of the most expensive appliance.

6. Code Cell An empty code cell.

7. Markdown Cell Contains a direct URL to an Amazon product (a TurboChef Microwave), likely a reference to the most expensive item found in the previous cell.

8. Code Cell Parses the raw JSON datapoints from the Hugging Face dataset into structured Item objects using the custom parse function. It filters out any None values (which occur if the item doesn't meet specific criteria like having a price between $1-$1000) and prints how many valid items were successfully parsed.

9. Code Cell Displays the details of the first parsed Item object (items[0]).

10. Code Cell Extracts two lists from the parsed items: prices (a list of all item prices) and lengths (a list of the character count of each item's full text description).

11. Code Cell Uses matplotlib to plot a histogram showing the distribution of the text lengths across all the appliance items, helping visualize the average and maximum lengths.

12. Code Cell Finds the item with the absolute longest text description and prints its complete text content.

13. Code Cell Plots another histogram, this time showing the distribution of prices across all appliance items (from $0 to $1000).

14. Code Cell Prints the full text description of the 4th item (items[3]).

15. Code Cell Introduces the ItemLoader class from pricer.loaders. It creates an instance for "Appliances" and uses its .load() method to load the items, overriding the manually parsed items list from earlier.

16. Code Cell Defines a list of 8 specific Amazon product categories (dataset_names) to be processed: Automotive, Electronics, Office Products, Tools & Home Improvement, Cell Phones, Toys & Games, Appliances, and Musical Instruments.

17. Code Cell Iterates over the dataset_names list, using the ItemLoader to load the items for every category, and aggregates them all into a single, massive items list.

18. Code Cell Prints the grand total of all items loaded across all the specified categories.

19. Code Cell Displays the details of the 1001st item (items[1000]) in the combined dataset.

20. Code Cell Performs dataset deduplication.

It first shuffles the dataset deterministically (using random.seed(42)).
It removes any duplicate items based on the product title.
It performs a second pass to remove duplicates based on the full text description.
Prints the total number of items remaining after cleaning.
21. Code Cell Plots a histogram of the text lengths for this new, massive, deduplicated multi-category dataset.

22. Code Cell Plots a histogram of the prices for the massive deduplicated dataset.

23. Code Cell Uses collections.Counter to count how many items belong to each product category. It then plots a bar chart to visualize this distribution, printing the exact count on top of each bar.

24. Code Cell Performs a complex, weighted random sampling to create a final dataset of exactly 820,000 items.

It normalizes the prices to calculate sampling weights (p**2), meaning more expensive items are given a significantly higher probability of being selected.
It artificially penalizes the weights for the "Tools_and_Home_Improvement" (cut in half) and "Automotive" (cut by 95%) categories to prevent them from dominating the sample.
It uses np.random.choice to pick the 820,000 items based on these calculated probabilities.
25. Code Cell Plots a price histogram of the newly sampled 820,000 items.

26. Code Cell Shuffles the final 820,000 item sample dataset one last time.

27. Code Cell Plots the exact same price histogram as cell 25 (likely left in by accident or just to double-check the distribution post-shuffle).

28. Code Cell Plots a bar chart showing the final category distribution in the sampled dataset of 820,000 items, verifying the effects of the manual weight penalties applied in Cell 24.

29. Code Cell Creates a donut chart (a pie chart with a white center) to show the percentage makeup of each category in the final sampled dataset.

30. Code Cell Creates a scatter plot comparing the size (text length) of an item on the X-axis to its price on the Y-axis to see if there is any visible correlation between how long a description is and how much the item costs.

31. Code Cell Creates a final scatter plot comparing the weight (in ounces) of an item on the X-axis to its price on the Y-axis to see if heavier items correlate with higher prices.

6:48 PM
